[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/course-2-generative-ai/adv-06-07-prompt-statistics-and-self-consistency.ipynb)

# Advanced Discussion 6 and 7 — comparing prompts statistically, and the mathematics of voting

Paired bootstrap and McNemar tests, power simulation for prompt evaluation, and majority-vote accuracy under independent, concentrated and question-dependent errors.

**Optional advanced-discussion lab** for Generative AI and Large Language Models with Python (AIBits Academy). Not graded and not needed for the certificate. Everything here runs on the Colab free tier; CPU is enough unless a note says otherwise.

Simulation only; runs in about a minute.

In [ ]:
import numpy as np
from math import comb
rng = np.random.RandomState(0)

## 1. is prompt B really better than prompt A? paired evaluation on a small test set

In [ ]:
def paired_bootstrap(a, b, iters=5000, seed=0):
    r = np.random.RandomState(seed); d = a.astype(float) - b.astype(float); n = len(d)
    means = np.array([d[r.randint(0, n, n)].mean() for _ in range(iters)]); return d.mean(), np.percentile(means, [2.5, 97.5])
def mcnemar_exact(a, b):
    n01 = int(((a == 0) & (b == 1)).sum()); n10 = int(((a == 1) & (b == 0)).sum()); n = n01 + n10
    if n == 0: return n01, n10, 1.0
    k = min(n01, n10); p = min(1.0, 2 * sum(comb(n, i) for i in range(0, k + 1)) / 2 ** n); return n01, n10, p
def simulate_pair(n, pA, pB, corr, seed):
    r = np.random.RandomState(seed); u = r.rand(n); v = corr * u + (1 - corr) * r.rand(n)      # correlated difficulty
    v = (np.argsort(np.argsort(v)) + 0.5) / n                                             # back to uniform
    return (u < pA).astype(int), (v < pB).astype(int)
a, b = simulate_pair(100, 0.72, 0.78, 0.8, 1)
print("100 items: accuracy A %.2f, B %.2f" % (a.mean(), b.mean()))
d, ci = paired_bootstrap(b, a); print("B - A = %.3f, 95%% CI [%.3f, %.3f]" % (d, *ci)); print("McNemar (B only right, A only right, p):", mcnemar_exact(a, b))
print("power to detect a true 6-point gain (A=0.72, B=0.78, paired correlation 0.8), 400 simulated experiments:")
for n in (50, 100, 300, 1000):
    hits = 0
    for s in range(400):
        aa, bb = simulate_pair(n, 0.72, 0.78, 0.8, 1000 + s)
        lo, hi = paired_bootstrap(bb, aa, iters=300, seed=s)[1]; hits += lo > 0
    print("  n = %4d items: CI excludes zero in %.0f%% of experiments" % (n, 100 * hits / 400))

## 2. self-consistency: majority vote over n samples

In [ ]:
def majority_binary(p, n):                                   # two possible answers; correct with prob p each sample, independent
    return sum(comb(n, k) * p ** k * (1 - p) ** (n - k) for k in range(n // 2 + 1, n + 1))
print("\nbinary answers, independent samples, per-sample accuracy p:")
print("  n     p=0.55   p=0.65   p=0.75   p=0.45")
for n in (1, 5, 11, 21, 41):
    print("  %-4d  %.3f    %.3f    %.3f    %.3f" % (n, *[majority_binary(p, n) for p in (0.55, 0.65, 0.75, 0.45)]))
def plurality(p, n, wrong_dist, trials=20000, seed=0):
    r = np.random.RandomState(seed); k = len(wrong_dist) + 1; probs = np.r_[p, (1 - p) * np.array(wrong_dist)]; ok = 0
    draws = r.choice(k, size=(trials, n), p=probs)
    for row in draws:
        c = np.bincount(row, minlength=k); ok += c.argmax() == 0
    return ok / trials
print("\nfree-form answers: per-sample accuracy 0.40 (wrong answers spread over 4 alternatives) vs concentrated on ONE wrong answer:")
print("  n     spread   concentrated")
for n in (1, 5, 11, 21, 41):
    print("  %-4d  %.3f    %.3f" % (n, plurality(0.40, n, [.25] * 4), plurality(0.40, n, [1.0])))
# question-level difficulty: p varies by question (some are just hard), voting cannot fix p<0.5 questions
diff_p = rng.beta(2, 1.4, 20000)                            # mean ~0.59, many hard questions
print("\nmean accuracy 0.59 but varying by question (Beta(2,1.4)):")
for n in (1, 5, 11, 21, 41):
    acc = np.mean([majority_binary(p, n) for p in diff_p[:1500]]); print("  n=%-3d majority-vote accuracy %.3f" % (n, acc))
print("  ceiling = share of questions with p > 0.5: %.3f" % (diff_p > 0.5).mean())

## Your turn

Change one setting at a time (seed, learning rate, ε, perplexity, threshold), re-run, and write one sentence about what changed and why. Keep the numbers: your results will differ slightly from the chapter's, and that difference is worth explaining.